In [1]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_ENDPOINT"]="false"

In [2]:
import sqlite3
from pathlib import Path
import pandas as pd

from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import (
    DoclingDocument,
    TextItem,
    TableItem,
)

def format_raw_pdf(input_doc_paths, output_dir: Path):
    output_dir = Path(output_dir)
    
    # 1. Configure v2 Pipeline Options
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = True
    pipeline_options.do_table_structure = True
    pipeline_options.table_structure_options.do_cell_matching = True

    # 2. Initialize DocumentConverter with PDF format options
    doc_converter = DocumentConverter(
        format_options={
            "pdf": PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    # 3. Convert all input paths (convert_all accepts a list of Path/str)
    conv_results = doc_converter.convert_all(input_doc_paths)

    # Prepare SQLite Database directory & connection
    db_folder_path = output_dir / "table" 
    db_file_path = db_folder_path / "all_tables.db"
    db_folder_path.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_file_path)

    table_id = 0
    print("Extraction Starting ...")

    for result in conv_results:
        # Access the parsed DoclingDocument
        doc: DoclingDocument = result.document
        file_stem = result.input.file.stem

        text_folder_path = output_dir / "text" 
        text_file_path = text_folder_path / f"{file_stem}.txt"
        text_folder_path.mkdir(parents=True, exist_ok=True)
        
        # Clear or create text file
        text_file_path.open("w", encoding="utf-8").close()

        prev_text = ""

        # 4. Iterate over items using Docling v2 tree iterator
        for item, _level in doc.iterate_items():
            
            # --- Text Handling ---
            if isinstance(item, TextItem):
                text = item.text.strip()
                if not text:
                    continue

                # Ignore repeated text
                if text == prev_text:
                    continue
                prev_text = text

                with text_file_path.open("a", encoding="utf-8") as f:
                    f.write(f"{text}\n\n")

            # --- Table Handling ---
            elif isinstance(item, TableItem):
                # Extract pandas DataFrame directly using built-in v2 exporter
                try:
                    table_df = item.export_to_dataframe()
                except Exception:
                    continue

                if table_df.empty:
                    continue

                # Skip Table of Contents if matched
                caption = item.caption.text if item.caption else ""
                if caption == "Table of Contents":
                    continue

                # Determine table name
                if caption:
                    # Sanitize caption for SQL table naming
                    table_name = "".join(c if c.isalnum() else "_" for c in caption)
                else:
                    table_name = f"table_{chr(ord('a') + table_id)}"
                    table_id += 1

                # Save table into SQLite database
                table_df.to_sql(name=table_name, con=conn, if_exists="replace", index=False)
                conn.commit()

    conn.close()
    print(f"Text and Tables are separated from the PDF files and stored in: {output_dir}")

c:\Users\Param\OneDrive\Desktop\MultiModal_RAG_Agent-main\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
input_doc_paths = [
    Path("./data/duplicate-chars.pdf")
]

output_dir = Path("./data/formatted/")

format_raw_pdf(input_doc_paths,output_dir)

### SQL Chain

Let's create a simple chain that takes a question, turns it into a SQL query, executes the query, and uses the result to answer the original question.

In [3]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///./data/formatted/table/all_tables.db")

C:\Users\Param\AppData\Local\Temp\ipykernel_14008\2477885037.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [4]:
from dotenv import load_dotenv

from langchain_community.utilities import SQLDatabase
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

# 1. Connect to SQLite database
db = SQLDatabase.from_uri("sqlite:///./data/formatted/table/all_tables.db")

# 2. Define your PromptTemplate
sql_gen_prompt = PromptTemplate(
    input_variables=['input', 'table_info'], 
    partial_variables={'top_k': '5'}, 
    template="""You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use date('now') function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result of the SQLQuery
Answer: Final answer here

Only use the following tables:
{table_info}

Question: {input}
SQLQuery: """
)

# 3. Initialize LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.7-flash", temperature=0)

# Helper function to clean markdown code fences from generated SQL
def clean_sql_output(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        cleaned = "\n".join(lines).strip()
    return cleaned

# 4. Build LCEL Chain
sql_chain = (
    {
        "table_info": lambda _: db.get_table_info(),
        "input": lambda x: x["question"],
    }
    | sql_gen_prompt
    | llm
    | StrOutputParser()
    | clean_sql_output
)

# 5. Initialize Execution Tool
execute_query = QuerySQLDataBaseTool(db=db)

# --- Run ---
question = "List all tables in the database"

# Step 1: Generate SQL query
sql_query = sql_chain.invoke({"question": question})
print(f"Generated SQL Query:\n{sql_query}\n")

# Step 2: Execute query against database
result = execute_query.invoke({"query": sql_query})
print(f"Query Result:\n{result}")

C:\Users\Param\AppData\Local\Temp\ipykernel_14008\2614509673.py:66: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  execute_query = QuerySQLDataBaseTool(db=db)
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Generated SQL Query:
SELECT "name" FROM sqlite_master WHERE "type"='table' LIMIT 5;
SQLResult: [('table_a',), ('table_b',), ('table_c',), ('table_d',)]
Answer: The tables in the database are table_a, table_b, table_c, and table_d.

Query Result:
Error: (sqlite3.ProgrammingError) You can only execute one statement at a time.
[SQL: SELECT "name" FROM sqlite_master WHERE "type"='table' LIMIT 5;
SQLResult: [('table_a',), ('table_b',), ('table_c',), ('table_d',)]
Answer: The tables in the database are table_a, table_b, table_c, and table_d.]
(Background on this error at: https://sqlalche.me/e/21/f405)


### RAG Chain

We will create a RAG chain with text documents

In [5]:
from langchain_community.document_loaders import TextLoader

def get_text_documents(text_dir_path):
    if os.path.isdir(text_dir_path):
        text_documents = []
        for file in os.listdir(text_dir_path):
            loader = TextLoader(text_dir_path+file)
            text_documents.extend(loader.load())
    else:
        loader = TextLoader(text_dir_path)
        text_documents = loader.load()
    
    return text_documents

In [6]:
from pathlib import Path
from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Utility to clean metadata for ChromaDB
from langchain_community.vectorstores.utils import filter_complex_metadata

# Gemini Integration Packages
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Docling LangChain Loader
from langchain_docling import DoclingLoader

load_dotenv()

# 1. Load documents using Docling Loader
file_paths = [str(p) for p in Path("data/formatted/text/").glob("*") if p.is_file()]
loader = DoclingLoader(file_path=file_paths)
docs = loader.load()

# 2. Chunk the documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# 3. Clean nested metadata from Docling for ChromaDB compatibility
cleaned_splits = filter_complex_metadata(splits)

# 4. Initialize Gemini Embeddings & Vector Store
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vectorstore = Chroma.from_documents(documents=cleaned_splits, embedding=embeddings)

# 5. Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.7-flash",
    temperature=0
)

# 6. Set up Retriever and Inline Prompt
retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "Question: {question}\n"
    "Context: {context}\n"
    "Answer:"
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 7. Build LCEL RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# --- Example Run ---
# response = rag_chain.invoke("What are the key takeaways from the document?")
# print(response)

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


### Multi-Modal Agent

In [ ]:
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent

load_dotenv()

# 1. Define Tools using @tool decorator
@tool
def sql_database(question: str) -> str:
    """Useful when you need to answer user questions from tabular or structured SQL data.
    Generates a SQL query, executes it against the database, and returns the query results.
    """
    # Step A: Generate raw SQL string
    generated_sql = sql_chain.invoke({"question": question}).strip()
    
    # Step B: Execute SQL string against SQLite database
    db_result = execute_query.invoke({"query": generated_sql})
    
    return f"Generated Query: {generated_sql}\nExecution Result: {db_result}"

@tool
def text_database(question: str) -> str:
    """Useful when you need to answer from the unstructured text document corpus.
    Use this for descriptive or informational questions.
    """
    return rag_chain.invoke(question)

@tool
def terminal_tool(command: str) -> str:
    """Executes a shell command in the local terminal and returns output.
    Use this to check files, inspect system info, or run shell commands.
    """
    import subprocess
    try:
        res = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=30)
        return res.stdout.strip() if res.stdout.strip() else res.stderr.strip()
    except Exception as e:
        return f"Error executing command: {str(e)}"

tools = [sql_database, text_database, terminal_tool]

# 2. Initialize Gemini Model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)

# 3. System Instruction
system_prompt = (
    "You are a helpful multi-modal assistant equipped with SQL database querying, "
    "document vector search, and local terminal execution capabilities. "
    "Use your tools when appropriate to answer questions accurately."
)

# 4. Construct Modern LangGraph Agent
# Replaces legacy AgentExecutor entirely
agent_executor = create_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

# --- Run Agent ---
inputs = {"messages": [("human", "What is Riken Keiki GP-1000?")]}

response = agent_executor.invoke(inputs)

# Extract and print the final assistant response
final_message = response["messages"][-1]
print(final_message.content)

C:\Users\Param\AppData\Local\Temp\ipykernel_14008\2069970187.py:59: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(
c:\Users\Param\OneDrive\Desktop\MultiModal_RAG_Agent-main\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\Param\OneDrive\Desktop\MultiModal_RAG_Agent-main\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The **Riken Keiki GP-1000** is a compact, lightweight, and high-sensitivity combustible gas detector (catalytic LEL gas detector). \n\nKey features include:\n* **Detection Principle:** Catalytic combustion.\n* **Detection Range:** 0–10% LEL or 0–100% LEL for hydrocarbons and other combustible gases (such as $\\text{CH}_4$).\n* **Features:**\n  * Built-in sampling pump with a pump booster function.\n  * Allows direct selection from a list of 25 combustible target gases.\n  * Only requires calibration on methane ($\\text{CH}_4$).', 'extras': {'signature': 'EmAKXgFpFH0TsPks8/OausdrUoQH/fUiioIyoTksXZ+6bPW5ysPtWjYG0ho9euTTZ0MK7iHFV18nBkJiIMo1W+aBTsBFA+BvsP8FJdJ10r3aUUzM2hFPTU0HVap1VN+1yJo='}}]
